# FINRL Walk-Forward Experiment

This notebook runs the direct-feature walk-forward experiment runner and visualizes portfolio performance against the S&P 500 / SPY benchmark.

Use small synthetic data locally. Use Colab for the full configured universe, direct portfolio optimization, and full walk-forward experiments.

In [ ]:
%cd /content
![ -d FINRL ] || git clone https://github.com/nidarshans/FINRL.git
%cd /content/FINRL

In [ ]:
%cd /content/FINRL
%pip install -e .


In [ ]:
# Keep notebook imports pointed at the live workspace package.

from datetime import date, timedelta

import polars as pl

from finrl.backtest.walk_forward import WalkForwardConfig
from finrl.dpo_jax import DPOConfig
from finrl.data import (
    MarketDataBundle,
    MarketDataConfig,
    UniverseConfig,
    build_rebalance_calendar,
    compute_open_to_open_returns,
    download_ohlcv,
)
from finrl.data.download import download_macro_series
from finrl.env.trading_env import EnvConfig
from finrl.experiments import (
    ExperimentConfig,
    RawExperimentData,
    build_allocation_figure,
    build_drawdown_figure,
    build_holdings_heatmap_granular,
    build_performance_figure,
    build_regime_portfolio_figure,
    build_spectral_figure,
    metrics_to_frame,
    run_walk_forward_experiment,
)
from finrl.features import FeatureConfig, build_feature_bundle, selected_direct_allocation_indices
from finrl.features.preprocessing import PreprocessingConfig
from finrl.features.schema import FeatureBundle
import numpy as np

## Prepared Data Contract

The runner expects prepared feature and return tables:

- `FeatureBundle` with asset and macro features plus a dummy 20-column spectral compatibility table.
- `returns`: Polars DataFrame with `decision_date` and one return column per tradable asset.
- `spy_returns`: Polars DataFrame with `decision_date` and `spy_return` for the same holding periods.

Replace the synthetic fixture below with the output of the data, feature, preprocessing, and return-preparation pipeline for full experiments. DPO trains the direct allocation head over explicitly routed asset features.

## Run With Real yfinance Data

Edit `TICKERS`, `START`, `END`, and `MAX_STOCKS`, then run this section in Colab. The code downloads real stock or bond ticker data plus SPY, computes daily open-to-open returns, builds causal per-asset features, and packages everything into `RawExperimentData` for the walk-forward runner.

In [ ]:
TICKERS = [
    "XLC", "XLY", "XLP", "XLE", "XLF",
    "XLV", "XLI", "XLB", "XLRE", "XLK", "XLU", 
    'GLD', 'MTUM', "IBB"
]
MAX_STOCKS = len(TICKERS)  # set to 100 after pasting your full universe
START = "2015-01-01"
END = "2026-07-25"
CACHE_DIR = "data/cache"
BENCHMARK_TICKER = "SPY"
REBALANCE_FREQUENCY = "daily"  # "daily" or "weekly"

universe = UniverseConfig(
    tickers=TICKERS,
    max_stocks=MAX_STOCKS,
    include_cash=False,
    benchmark_ticker=BENCHMARK_TICKER,
)
market_config = MarketDataConfig(
    universe=universe,
    start=START,
    end=END,
    cache_dir=CACHE_DIR,
)
selected_tickers = universe.selected_tickers
selected_tickers

In [ ]:
def _returns_wide(open_to_open_returns: pl.DataFrame, tickers: tuple[str, ...]) -> pl.DataFrame:
    wide = (
        open_to_open_returns
        .select(["decision_date", "ticker", "return"])
        .pivot(index="decision_date", on="ticker", values="return", aggregate_function="first")
        .sort("decision_date")
    )
    return wide.select(["decision_date", *tickers]).fill_null(0.0)


def _spy_returns(open_to_open_returns: pl.DataFrame) -> pl.DataFrame:
    return (
        open_to_open_returns
        .select(["decision_date", pl.col("return").alias("spy_return")])
        .sort("decision_date")
        .drop_nulls()
    )


def _filter_features_to_common_dates(features: FeatureBundle, returns: pl.DataFrame, spy_returns: pl.DataFrame) -> FeatureBundle:
    common_dates = (
        returns.select("decision_date")
        .join(spy_returns.select("decision_date"), on="decision_date", how="inner")
        .rename({"decision_date": "date"})
        .with_columns(pl.col("date").cast(pl.Date))
        .unique()
        .sort("date")
    )
    asset = features.asset_features.join(common_dates, on="date", how="inner").sort(["date", "ticker"])
    macro = (
        common_dates
        .join(features.macro_features, on="date", how="left")
        .sort("date")
        .with_columns(pl.all().exclude("date").forward_fill().fill_null(0.0))
    )
    spectral = features.spectral_features.join(common_dates, on="date", how="inner").sort("date")
    dates = tuple(common_dates.get_column("date").to_list())
    return FeatureBundle(
        asset_features=asset,
        macro_features=macro,
        spectral_features=spectral,
        decision_dates=dates,
        tickers=features.tickers,
        asset_feature_columns=features.asset_feature_columns,
        macro_feature_columns=features.macro_feature_columns,
        spectral_feature_columns=features.spectral_feature_columns,
    )


def make_real_yfinance_data() -> RawExperimentData:
    ohlcv = download_ohlcv(selected_tickers, START, END, market_config)
    spy_ohlcv = download_ohlcv((BENCHMARK_TICKER,), START, END, market_config)
    macro = download_macro_series(START, END, market_config)
    calendar = build_rebalance_calendar(ohlcv, REBALANCE_FREQUENCY)

    market_bundle = MarketDataBundle(
        ohlcv=ohlcv,
        spy_ohlcv=spy_ohlcv,
        macro=macro,
        calendar=calendar,
    )
    features = build_feature_bundle(
        market_bundle,
        FeatureConfig(
            accumulation_window=40,
            klinger_fast_span=34,
            klinger_slow_span=55,
            klinger_signal_span=13,
            macd_fast_span=12,
            macd_slow_span=26,
            macd_signal_span=9,
            mr_ewma_span=200,
            mr_vol_window=200,
            spectral_dim=20,
            cmf_window=60,
        ),
    )

    stock_returns = _returns_wide(
        compute_open_to_open_returns(ohlcv, calendar),
        selected_tickers,
    )
    spy_returns = _spy_returns(compute_open_to_open_returns(spy_ohlcv, calendar))
    features = _filter_features_to_common_dates(features, stock_returns, spy_returns)
    common_dates = pl.DataFrame({"decision_date": list(features.decision_dates)}).with_columns(pl.col("decision_date").cast(pl.Date))
    stock_returns = common_dates.join(stock_returns, on="decision_date", how="inner")
    spy_returns = common_dates.join(spy_returns, on="decision_date", how="inner")
    return RawExperimentData(features=features, returns=stock_returns, spy_returns=spy_returns)


raw_data = make_real_yfinance_data()
raw_data.features.asset_features.tail(), raw_data.returns.tail(), raw_data.spy_returns.tail()

In [ ]:
n_stocks = len(raw_data.features.tickers)
n_tradable_assets = n_stocks + 1  # risky assets plus cash
asset_feature_dim = len(raw_data.features.asset_feature_columns)
macro_feature_dim = len(raw_data.features.macro_feature_columns)
routing = selected_direct_allocation_indices(raw_data.features.asset_feature_columns)

# DPO uses one complete chronological scan per epoch; there is no batch-size setting.
# It must execute the same dense allocations used by its training objective.
# Top-N execution is available only for the environment-only path.
# DPO maximizes net-return Sharpe ratio; transaction costs are disabled.
POLICY_MODE = "dpo"  # "dpo" or "equal_weight"
TOP_N_POSITIONS = None

config = ExperimentConfig(
    walk_forward=WalkForwardConfig(train_years=3, test_years=1, step_years=1),
    preprocessing=PreprocessingConfig(rolling_window=252),
    dpo=DPOConfig(
        learning_rate=1e-4,
        num_epochs=50,
        transaction_cost_bps=0.0,
        allocation_hidden_dims=(16, 8),
        allocation_hidden_activation="tanh",
        allocation_output_activation="identity",
        allocation_use_layer_norm=True,
        simplex_activation="sparsemax",
    ),
    env=EnvConfig(
        sortino_target_return=0.0,
        sortino_downside_penalty=0.0,
        top_n_positions=TOP_N_POSITIONS,
        transaction_cost_rate=0.0
    ),
    enable_dpo=POLICY_MODE == "dpo",
    rebalance_frequency=REBALANCE_FREQUENCY,
    seed=6,
)

print({
    "stocks": n_stocks,
    "tradable_assets": n_tradable_assets,
    "asset_feature_dim": asset_feature_dim,
    "macro_feature_dim": macro_feature_dim,
    "policy_mode": POLICY_MODE,
    "top_n_positions": TOP_N_POSITIONS,
    "direct_features": len(routing.direct_allocation_indices),
    "decision_dates": len(raw_data.features.decision_dates),
})

result = run_walk_forward_experiment(raw_data, config)
metrics_to_frame(result)

## Performance vs S&P 500


In [ ]:
performance_fig = build_performance_figure(result)
performance_fig.show()


## Drawdown

Compare portfolio and SPY peak-to-trough declines over the walk-forward period.


In [ ]:
drawdown_fig = build_drawdown_figure(result)
drawdown_fig.show()


## Portfolio Allocation


In [ ]:
allocation_fig = build_allocation_figure(result)
allocation_fig.show()


## Holdings Heatmap


In [ ]:
holdings_heatmap_fig = build_holdings_heatmap_granular(
    result,
    min_weight=0.001,
    top_n=50,
    freq=None,
    include_cash=False,
)
holdings_heatmap_fig.show()
